# Global M2 Aggregator + Bitcoin Price Prediction

End-to-end pipeline that:
1. Loads BTC price history (Yahoo Finance) and Global M2 liquidity (FRED API)
2. Engineers features and a binary target (`up_7d`)
3. Splits data chronologically (train / val / test)
4. Trains a Stacked Denoising Autoencoder (SDAE) + LightGBM classifier
5. Evaluates the model and displays charts inline
6. Produces a next-period BTC directional forecast

> **Running in Google Colab?**  
> Execute the **Setup** cell first – it clones the repository and installs all dependencies.

## Setup (Google Colab only)

This cell detects whether you are running in Colab and, if so:
- Clones the repository
- Changes the working directory into it
- Installs all Python dependencies

If you are running **locally** (Jupyter Lab / Notebook) with the repository already on disk, skip this cell and make sure dependencies are installed (`pip install -r requirements.txt`).

In [ ]:
import sys
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone the repository (skip if already present)
    if not os.path.isdir("m2-aggregator-global-liquidity"):
        !git clone https://github.com/alexanderbkl/m2-aggregator-global-liquidity.git
    %cd m2-aggregator-global-liquidity
    # Install dependencies inside Colab's ephemeral environment
    !pip install -r requirements.txt -q

## Configuration

The `CONFIG` dict below mirrors `config.py`.  
Edit any value here **before** running the pipeline cells – changes take effect immediately without restarting the kernel.

| Key | What it controls |
|---|---|
| `fred_api_key` | Required when `m2_source == "bis_fred"` – get a free key at https://fred.stlouisfed.org/docs/api/api_key.html |
| `use_m2_exog` | `True` to include Global M2 features; `False` for a BTC-only ablation run |
| `m2_source` | `"bis_fred"` (default) or `"csv"` (pre-computed file) |
| `btc_target_horizon` | Number of days ahead to predict direction (default: 7) |
| `output_dir` | Folder where chart PNGs are saved |

In [ ]:
import os

CONFIG = {
    # ── BTC data ──────────────────────────────────────────────────────────────
    "btc_ticker": "BTC-USD",
    "btc_start_date": "2014-01-01",
    "btc_target_horizon": 7,

    # ── Global M2 feature flag ────────────────────────────────────────────────
    "use_m2_exog": True,

    # ── M2 data source ────────────────────────────────────────────────────────
    # "bis_fred" → fetch from FRED API
    # "csv"      → load from a pre-computed CSV at m2_csv_path
    "m2_source": "bis_fred",
    "m2_csv_path": os.path.join("data", "global_m2.csv"),

    # FRED API key – paste your key here or set the FRED_API_KEY env variable
    "fred_api_key": os.environ.get("FRED_API_KEY", "aaf3121388bab2aba7ad45a91c0790a4"),

    # Country basket for M2 aggregation
    "m2_countries": ["US", "EA", "CN", "JP", "GB"],

    # Lag offsets (calendar days) for the lagged M2 features
    "m2_lag_days": [1, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98],

    # Growth windows (days) computed on the daily M2 series
    "m2_growth_windows": [7, 30, 90],

    # ── Feature engineering ───────────────────────────────────────────────────
    "btc_rolling_windows": [7, 14, 30, 60, 90],

    # ── Train / validation / test split ──────────────────────────────────────
    "train_ratio": 0.70,
    "val_ratio": 0.15,

    # ── SDAE (Stacked Denoising Autoencoder) ──────────────────────────────────
    "sdae_hidden_dims": [256, 128, 64],
    "sdae_noise_factor": 0.1,
    "sdae_dropout": 0.2,
    "sdae_learning_rate": 1e-3,
    "sdae_weight_decay": 1e-5,
    "sdae_epochs": 50,
    "sdae_batch_size": 256,
    "sdae_patience": 10,
    "sdae_log_every_epochs": 1,
    "sdae_log_every_batches": 0,
    "sdae_torch_num_threads": 2,

    # ── LightGBM ──────────────────────────────────────────────────────────────
    "lgbm_params": {
        "objective": "binary",
        "metric": "binary_logloss",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "num_leaves": 63,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    },
    "lgbm_early_stopping_rounds": 50,

    # ── Evaluation & output ───────────────────────────────────────────────────
    "output_dir": "outputs",
    "save_plots": True,

    # Regime analysis column
    "regime_column": "m2_90d_chg",
    "regime_n_bins": 3,
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Configuration loaded.")
print(f"  use_m2_exog      : {CONFIG['use_m2_exog']}")
print(f"  m2_source        : {CONFIG['m2_source']}")
print(f"  btc_target_horizon: {CONFIG['btc_target_horizon']} days")
print(f"  output_dir       : {CONFIG['output_dir']}")

## Imports

In [ ]:
import logging
import sys

import numpy as np
import pandas as pd

# Use inline matplotlib backend before importing pipeline modules
import matplotlib
matplotlib.use("Agg")  # headless backend – plots are saved to disk then displayed below
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pipeline.data_loader import load_data
from pipeline.features import build_features_and_target, _add_btc_features, _add_target
from pipeline.model import predict, train_model
from pipeline.evaluation import (
    compute_metrics,
    plot_confusion_matrix,
    plot_equity_curve,
    plot_feature_importance,
    plot_regime_accuracy,
    plot_shap_beeswarm,
)
from pipeline.sdae import encode_features

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s – %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("notebook")
print("Imports OK.")

## Helper: chronological split

In [ ]:
def _split(X, y, df_full, config):
    """Chronological train / val / test split (no shuffling)."""
    n = len(X)
    n_train = int(n * config["train_ratio"])
    n_val   = int(n * config["val_ratio"])

    X_train, y_train = X.iloc[:n_train],              y.iloc[:n_train]
    X_val,   y_val   = X.iloc[n_train:n_train+n_val], y.iloc[n_train:n_train+n_val]
    X_test,  y_test  = X.iloc[n_train+n_val:],        y.iloc[n_train+n_val:]

    df_test_rows = (
        df_full
        .iloc[n_train+n_val : n_train+n_val+len(X_test)]
        .reset_index(drop=True)
    )

    logger.info("Split  train=%d  val=%d  test=%d", len(X_train), len(X_val), len(X_test))
    return X_train, y_train, X_val, y_val, X_test, y_test, df_test_rows

## Step 1 – Load data

Downloads BTC OHLCV from Yahoo Finance.  
When `use_m2_exog=True`, also fetches the Global M2 series from FRED and inner-joins it to the BTC frame.

In [ ]:
logger.info("=== Step 1: Load data ===")
df = load_data(CONFIG)

if df.empty:
    raise RuntimeError("No data loaded. Check your FRED API key or network connectivity.")

print(f"Loaded {len(df):,} rows  |  columns: {list(df.columns[:10])} …")
df.tail(3)

## Step 2 – Feature engineering

Computes BTC technical features (log-returns, rolling z-scores, normalised range, volume change) and, when enabled, all M2 lag / growth features. Also creates the binary target column (`up_7d` by default).

In [ ]:
logger.info("=== Step 2: Build features ===")
X, y, feature_cols = build_features_and_target(df, CONFIG)

if len(X) < 200:
    raise RuntimeError(
        f"Insufficient rows ({len(X)}) after feature engineering. "
        "Extend btc_start_date or check the M2 data coverage."
    )

# Rebuild aligned df to preserve regime columns needed for evaluation
horizon     = CONFIG.get("btc_target_horizon", 7)
target_col  = f"up_{horizon}d"
df_aligned  = df.copy()
df_aligned  = _add_btc_features(df_aligned, CONFIG)
df_aligned  = _add_target(df_aligned, CONFIG)
df_aligned  = df_aligned.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

print(f"Features : {len(feature_cols)}")
print(f"Samples  : {len(X)}")
print(f"Target   : {target_col}  (class balance: {y.mean():.2%} up)")

## Step 3 – Chronological train / val / test split

In [ ]:
logger.info("=== Step 3: Split ===")
X_train, y_train, X_val, y_val, X_test, y_test, df_test_rows = _split(
    X, y, df_aligned, CONFIG
)
print(f"Train : {len(X_train):>5,} rows")
print(f"Val   : {len(X_val):>5,} rows")
print(f"Test  : {len(X_test):>5,} rows")

## Step 4 – Train SDAE + LightGBM

1. `StandardScaler` normalises the feature matrix.
2. The SDAE encoder compresses features to a 64-dimensional latent vector.
3. Raw M2 features are appended to the latent vector.
4. LightGBM trains on the resulting representation with early stopping on the validation set.

> This step may take several minutes on CPU.

In [ ]:
logger.info("=== Step 4: Train SDAE + LightGBM ===")
sdae_model, scaler, lgbm_model, lgbm_features = train_model(
    X_train, y_train, X_val, y_val, feature_cols, CONFIG
)
print("Training complete.")
print(f"  LightGBM features : {len(lgbm_features)}")
print(f"  Best iteration    : {lgbm_model.best_iteration_}")

## Step 5 – Evaluate

Runs the model on the held-out test set and displays all evaluation charts.

In [ ]:
logger.info("=== Step 5: Evaluate ===")
p_up    = predict(X_test, feature_cols, sdae_model, scaler, lgbm_model, CONFIG)
metrics = compute_metrics(y_test.values, p_up)

print("\n── Test-set metrics ──────────────────")
for k, v in metrics.items():
    print(f"  {k:<20}: {v:.4f}")

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────────
plot_confusion_matrix(y_test.values, p_up, CONFIG)
display(Image(os.path.join(CONFIG["output_dir"], "confusion_matrix.png")))

In [ ]:
# ── Equity curve ───────────────────────────────────────────────────────────────
plot_equity_curve(y_test.values, p_up, CONFIG)
display(Image(os.path.join(CONFIG["output_dir"], "equity_curve.png")))

In [ ]:
# ── Feature importance ─────────────────────────────────────────────────────────
plot_feature_importance(lgbm_model, lgbm_features, CONFIG)
display(Image(os.path.join(CONFIG["output_dir"], "feature_importance.png")))

m2_imp_path = os.path.join(CONFIG["output_dir"], "m2_feature_importance.png")
if os.path.isfile(m2_imp_path):
    display(Image(m2_imp_path))

In [ ]:
# ── SHAP beeswarm ──────────────────────────────────────────────────────────────
X_test_sc  = scaler.transform(X_test[feature_cols].values)
Z_test     = encode_features(sdae_model, X_test_sc)

use_m2  = CONFIG.get("use_m2_exog", True)
m2_cols = (
    [c for c in feature_cols if c.startswith("M2_") or c.startswith("m2_")]
    if use_m2 else []
)
if m2_cols:
    m2_idx     = [feature_cols.index(c) for c in m2_cols]
    Z_test_full = np.hstack([Z_test, X_test_sc[:, m2_idx]])
else:
    Z_test_full = Z_test

plot_shap_beeswarm(lgbm_model, Z_test_full, lgbm_features, CONFIG)

shap_path = os.path.join(CONFIG["output_dir"], "shap_beeswarm.png")
if os.path.isfile(shap_path):
    display(Image(shap_path))

In [ ]:
# ── Regime accuracy ────────────────────────────────────────────────────────────
plot_regime_accuracy(df_test_rows, y_test.values, p_up, CONFIG)

regime_path = os.path.join(CONFIG["output_dir"], "m2_regime_accuracy.png")
if os.path.isfile(regime_path):
    display(Image(regime_path))

## Step 6 – Predict next period

Uses the most recent available BTC + M2 row to produce a directional forecast for the next `btc_target_horizon` days.

In [ ]:
logger.info("=== Step 6: Predict next period ===")
last_row = X.tail(1)
p_next   = predict(last_row, feature_cols, sdae_model, scaler, lgbm_model, CONFIG)
signal   = "UP ↑" if p_next[0] >= 0.5 else "DOWN ↓"

print("\n" + "═" * 45)
print(f"  Next {horizon}-day BTC forecast")
print(f"  p_up   : {p_next[0]:.4f}")
print(f"  Signal : {signal}")
print("═" * 45)

results = {
    "metrics"             : metrics,
    "next_period_p_up"    : float(p_next[0]),
    "next_period_signal"  : signal,
    "feature_count"       : len(feature_cols),
    "m2_feature_count"    : len(m2_cols),
}
print("\nFull results dict:")
results